In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.isdir("smartstock-ai"):          # Colab/Kaggle: clone your repo first
    subprocess.run(["git", "clone", "https://github.com/<you>/smartstock-ai.git"], check=True); os.chdir("smartstock-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements.txt"], check=True)
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "backend")) and ROOT != "/": ROOT = os.path.dirname(ROOT)
os.chdir(ROOT); sys.path[:0] = [os.path.join(ROOT, "backend"), os.path.join(ROOT, "training", "scripts")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RAW_OK = os.path.exists("data/raw/m5/sales_train_validation.csv")
if not RAW_OK:   # no real M5 download present -> generate M5-format synthetic data so the notebook runs anywhere
    subprocess.run([sys.executable, "training/scripts/make_synthetic_m5.py"], check=True)
if not os.path.exists("data/processed/panel.npz"):
    subprocess.run([sys.executable, "training/scripts/prepare_data.py"], check=True)
def ensure_trained():
    if not os.path.exists("training/outputs/metrics.json"):
        subprocess.run([sys.executable, "training/scripts/train.py"], check=True)

# 02 · EDA — business-focused views only

In [ ]:
d = pd.read_parquet("data/processed/sales_daily.parquet"); d["dow"] = d.date.dt.day_name()
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
d.groupby(d.date.dt.dayofweek).sales.mean().plot.bar(ax=ax[0], title="Avg units by weekday (0=Mon)")
d.groupby([d.date.dt.to_period("M"), "cat_id"]).sales.sum().unstack().plot(ax=ax[1], title="Monthly units by category")
z = d.groupby(["item_id", "store_id"]).sales.apply(lambda s: (s == 0).mean()); z.hist(bins=20, ax=ax[2]); ax[2].set_title("Share of zero-sales days per series (intermittency)")
plt.tight_layout(); plt.show()

In [ ]:
d["promo"] = d.sell_price < 0.92 * d.groupby(["item_id", "store_id"]).sell_price.transform("mean")
print(d.groupby(["cat_id", "promo"]).sales.mean().unstack().round(2).rename(columns={False: "regular", True: "promo"}))
print("event days vs normal:", d.groupby(d.event_name_1.fillna("") != "").sales.mean().round(2).to_dict())